# EMFF N-State Fit Template

This notebook is a template wrapper around the repository's EMFF ratio-fit workflow for pion electromagnetic form factors.
Edit the input block below, validate it, and then run the same backend used by the CLI.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

# --- Set REPO_ROOT to the lat-hadron-analysis repository ---
REPO_ROOT = None
for _candidate in [
    Path.cwd().resolve(),
    Path('/Users/xiang/Desktop/codes/lat-hadron-analysis'),  # <-- EDIT THIS
]:
    if _candidate is not None and (_candidate / 'src' / 'lqcd_analysis').is_dir():
        REPO_ROOT = _candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Cannot find lat-hadron-analysis repository. '
        'Edit the hard-coded path in the cell above.'
    )
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f'REPO_ROOT: {REPO_ROOT}')

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_emff_fit_input_text,
    run_emff_fit_from_notebook,
    validate_emff_notebook_config,
)


## User Inputs

These fields mirror the plain-text EMFF input file format.
Point the paths at your HDF5 3pt data, HDF5 2pt data, and existing two-point n-state fit results.


In [ ]:
# --- Edit these paths for your analysis ---
DATA_3PT = Path("/Users/xiang/Desktop/docs/0-2026/projects/3-EMFF/data/l48c64a060_m140/pion_EMFF")
DATA_2PT = Path("/Users/xiang/Desktop/docs/0-2026/projects/3-EMFF/data/l48c64a060_m140/c2pt")
RESULTS_2PT = Path("/Users/xiang/Desktop/docs/0-2026/projects/3-EMFF/data/l48c64a060_m140/2pt_results")
WINDOW_2PT = RESULTS_2PT / "2pt_fit_windows.txt"
OUTPUT_DIR = Path("results_emff_notebook")

workflow_config = {
    # --- Title and lattice ---
    "title_pattern": "l48c64a060_m140_EMFF",
    "ns": 48,
    "nt": 64,
    "lattice_spacing_fm": 0.060,
    "hadron_mass_gev": 0.140,

    # --- Operators (must match 2pt data) ---
    "src_gamma": "5",
    "sink_gamma": "5",
    "insert_gamma": "T",   # vector current gamma_t for EMFF

    # --- Data paths (use str() for Path objects) ---
    "c2pt": str(DATA_2PT / "l48c64a060.c2pt.CFG.EMFF.ex.SRC.1HYP_M140_GSRC_W52_5.posSrc000_posSink000_negSrc000_negSink000.src{src_gamma}.h5"),
    "c3pt_h5": str(DATA_3PT / "l48c64a060.pion_EMFF.CFG.EMFF.ex.SRC.1HYP_M140_GSRC_W52_5.posSrc000_posSink000_negSrc000_negSink000.src{src_gamma}.PX{pfx}PY{pfy}PZ{pfz}dt{tsep}.h5"),
    "c3pt_dataset_path": "SS/{insert_gamma}/PX{qx}PY{qy}PZ{qz}",

    # --- Two-point fit reference ---
    "two_point_fit_root": str(RESULTS_2PT),
    "two_point_fit_window_by_pz": {0: [8, 20]},

    # --- Momentum selection ---
    "pflist": [0, 0, 0],          # final momentum Pf: pfx pfy pfz
    "qxlist": [-2, -1, 0, 1, 2],  # momentum transfer qx
    "qylist": [-2, -1, 0, 1, 2],  # momentum transfer qy
    "qzlist": [-2, -1, 0, 1, 2],  # momentum transfer qz
    "average_transverse_orbits": True,

    # --- Time separation ---
    "tslist": [4, 6, 8, 10, 12],

    # --- Fit settings ---
    "fit_method": "2state",        # "2state", "summation", or "plateau"
    "nstates": [2],                # 1 and/or 2
    "tau_range": [1, -1],          # tau from 1 to tsep-1
    "tsep_range": [4, 12],         # tsep fitting window

    # --- Bootstrap ---
    "binsize": 1,
    "bootstrap_samples": "auto",   # "auto" or integer
    "bootstrap_size": "auto",
    "seed": 2026,

    # --- Output ---
    "plot": True,
    "results_dir": str(OUTPUT_DIR),
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.

### Data Settings
- `title_pattern`: Output title. Used for naming output directories and matching 2pt fit titles.
- `ns`, `nt`: Spatial and temporal lattice extents.
- `lattice_spacing_fm`: Lattice spacing in fm.
- `hadron_mass_gev`: Hadron mass used in the dispersion relation for the ratio energy prefactor.

### Operator Settings
- `src_gamma`: Source gamma matrix (e.g., `"5"` for gamma_5). Must match the 2pt HDF5 file name.
- `sink_gamma`: Sink gamma matrix. Must match the `SS/<sink_gamma>/` group in the 2pt HDF5 file.
- `insert_gamma`: Current insertion gamma. Available under `SS/` in HDF5: `5`, `I`, `T`, `X`, `Y`, `Z`, `T5`, `X5`, `Y5`, `Z5`, `SXT`, `SXY`, `SXZ`, `SYT`, `SYZ`, `SZT`. For pion EMFF vector current, use `"T"` (gamma_t).

### Correlator Paths
- `c2pt`: Two-point HDF5 file path template. The workflow reads `SS/{sink_gamma}/PX{pfx}PY{pfy}PZ{pfz}` directly from this file. Placeholders: `{src_gamma}`, `{sink_gamma}`, `{pz}`, `{pfx}`, `{pfy}`, `{pfz}`.
- `c3pt_h5`: Three-point HDF5 file path/glob template. Placeholders: `{src_gamma}`, `{pfx}`, `{pfy}`, `{pfz}`, `{tsep}`.
- `c3pt_dataset_path`: Three-point internal HDF5 dataset path template. Placeholders: `{insert_gamma}`, `{qx}`, `{qy}`, `{qz}`.

### Momentum Selection
- `pflist`: Final momentum Pf as `[pfx, pfy, pfz]`. The ratio uses `P_i = P_f - q` and reads both initial/final 2pt datasets.
- The ratio follows Eq. (6) of arXiv:2102.06047 with `E(P)=sqrt(hadron_mass_gev^2 + P^2)`.
- `average_transverse_orbits`: When true, average degenerate transverse sign flips and `x/y` exchanges before building the ratio, and save only canonical non-negative representatives. This requires `Pf_x = Pf_y = 0`.
- `qxlist`, `qylist`, `qzlist`: Momentum transfer values. Can use `qxrange`/`qyrange`/`qzrange` as `[start, stop]`.
- Total q combinations = len(qxlist) x len(qylist) x len(qzlist).


### Ratio Formula
The workflow builds the ratio using Eq. (6) of arXiv:2102.06047:

```text
R^{fi}(tau, ts)
= [2 sqrt(Ef Ei) / (Ef + Ei)]
  * C3pt(Pf, Pi, tau, ts) / C2pt(ts, Pi)
  * sqrt[
      C2pt(ts - tau, Pf) C2pt(tau, Pi) C2pt(ts, Pi)
      /
      C2pt(ts - tau, Pi) C2pt(tau, Pf) C2pt(ts, Pf)
    ]
```

Here `q = Pf - Pi`, so `Pi = Pf - q`. Energies use the input hadron mass:

```text
E(P) = sqrt(hadron_mass_gev^2 + |P|^2),
P = 2 pi n / (a Ns).
```

When transverse orbit averaging is enabled, the workflow first averages the degenerate transverse orbit members at the correlator level (`C3pt` and the corresponding initial-state `C2pt`) and then applies the ratio formula to the averaged correlators. It does not average `qz` with `-qz`.

### Time Separation
- `tslist`: Time separations to load from HDF5 files.
- `tsep_range`: `[tsep_min, tsep_max]` — which tsep values participate in the fit.
- `tau_range`: `[tau_min, tau_offset]`. `tau_offset = -1` means tau up to tsep-1 per tsep.

### Fit Settings
- `fit_method`:
  - `"2state"` — 2-state model: R = (M_00 + M_01 e^{-dE tau} + M_10 e^{-dE (tsep-tau)} + M_11 e^{-dE tsep}) / (1 + R_1 e^{-dE tsep})
  - `"summation"` — Sum over tau: S(tsep) = tsep * M_00 + B
  - `"plateau"` — Constant: R = M_00
- `nstates`: `1` or `2` or `[1, 2]`. For 2-state with nstates=1, only M_00 is fitted.
- The fit uses the real part of the complex ratio.

### Two-Point Fit Reference
- `two_point_fit_root`: Root directory of 2pt nstate fit outputs.
- `two_point_fit_window_by_pz`: Dict `{pz: [tmin, tmax]}` for 2pt fit windows.
- Energies (E_0, E_1) and amplitudes (A_0, A_1) from 2pt fits provide DeltaE and R_1.

### Bootstrap
- `binsize`: Bin size (default 1).
- `bootstrap_samples`, `bootstrap_size`: `"auto"` or integer.
- `seed`: Random seed.

### Output Files (per q value)
- `{title}_{gamma}_q{+qx}_{+qy}_{+qz}_ratio.txt` — Ratio data at all (tsep, tau)
- `{title}_{gamma}_q{+qx}_{+qy}_{+qz}_{n}state_summary.txt` — Fit summary
- `{title}_{gamma}_q{+qx}_{+qy}_{+qz}_{n}state_fit.txt` — Fit table
- `{title}_{gamma}_q{+qx}_{+qy}_{+qz}_{n}state_samples.txt` — Bootstrap samples


## Validate Config


In [ ]:
parsed = validate_emff_notebook_config(workflow_config)
parsed


## Render Plain-Text Input Preview


In [ ]:
print(render_emff_fit_input_text(workflow_config))


## Run Backend Workflow


In [ ]:
outputs = run_emff_fit_from_notebook(workflow_config)
for output in outputs:
    print(output)


## Config Snapshot


In [ ]:
print(pretty_print_config(workflow_config))
